# Recepten, chunking en retrieval

## Waarom deze aanpak?
De dataset bevat dertig synthetische voorbeelden, niet gecontroleerde echte recepten. Elke rij bevat naam, porties, ingrediënten, stappen en een dieetlabel. Bereidingstijden zijn nog onbekend. Controleer recepten en allergenen voordat je inhoudelijke conclusies trekt.

We maken per recept vier afzonderlijke stukken: overzicht, ingrediënten, instructies en dieetlabels. Zo kan een vraag over ingrediënten gericht een ingrediëntenfragment vinden. We voegen de receptnaam en sectie als metadata toe; een nadeel is dat samenhang tussen delen verloren kan gaan. Een parent-child-aanpak is een mogelijk experiment. Het embeddingmodel draait lokaal en Chroma slaat de vectoren en originele tekst op. De collectie wordt momenteel elke start opnieuw opgebouwd; dit is eenvoudig maar kost tijd en is geen productieaanpak.

De zoekopdracht haalt vijf fragmenten op. Een hogere waarde kan recall verhogen maar levert meer ruis en tokens op. Vergelijk dit met andere waarden op een vaste evaluatieset in plaats van aan te nemen dat vijf optimaal is.


## Uitvoerbare implementatie
Voer deze cel uit vanuit de projectmap om de uitvoerbare versie bij te werken.


In [3]:
%%writefile chef/retrieval.py
"""Build and query a local recipe index."""
import json
import os
from pathlib import Path
import chromadb
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

ROOT = Path(__file__).resolve().parent.parent

def build_index():
    recipes = json.loads((ROOT / "data" / "recipes.json").read_text(encoding="utf-8"))
    documents = []
    for recipe in recipes:
        common = {"recipe": recipe["name"], "servings": recipe["servings"]}
        for section in ("overview", "ingredients", "instructions", "dietary_tags"):
            value = recipe[section]
            text = json.dumps(value, ensure_ascii=False) if isinstance(value, list) else value
            documents.append(Document(text=f"Recipe: {recipe['name']}\n{section}: {text}", metadata={**common, "section": section}))
    client = chromadb.PersistentClient(path=str(ROOT / "storage" / "chroma"))
    # Rebuild the collection to prevent duplicate entries on repeated starts.
    if any(collection.name == "recipes" for collection in client.list_collections()):
        client.delete_collection("recipes")
    collection = client.get_or_create_collection("recipes")
    store = ChromaVectorStore(chroma_collection=collection)
    embedding = HuggingFaceEmbedding(model_name=os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"), cache_folder=str(ROOT / "storage" / "models"))
    return VectorStoreIndex.from_documents(documents, storage_context=StorageContext.from_defaults(vector_store=store), embed_model=embedding)


Overwriting chef/retrieval.py
